# 14 — Full Market Scan And Watchlist

Notebook nay da chuyen sang workflow moi:

- universe full market duoc quet nhe tu notebook 02
- notebook 02 se tu dong tao `auto top 10 watchlist`
- notebook 14 uu tien scan sau tren `auto top 10` nay, nhung van cho phep ban override thu cong

Muc tieu: khong tinh toan nang tren toan bo thi truong neu chua can thiet.


In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path

def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")

project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Screening Modules
import importlib
import inspect

import pandas as pd

import src.reporting.chart_factory as chart_factory
import src.screening.stealth_accumulation as stealth_accumulation
from src.data.loaders import read_parquet, save_parquet
from src.screening.light_market_scan import load_auto_watchlist_symbols

chart_factory = importlib.reload(chart_factory)
stealth_accumulation = importlib.reload(stealth_accumulation)

load_screen_config = stealth_accumulation.load_screen_config
run_stealth_accumulation_screen = stealth_accumulation.run_stealth_accumulation_screen
latest_screen_snapshot = stealth_accumulation.latest_screen_snapshot
compare_screen_backtests = stealth_accumulation.compare_screen_backtests
entropy_regime_chart = chart_factory.entropy_regime_chart
screen_score_bar_chart = chart_factory.screen_score_bar_chart
stage_distribution_chart = chart_factory.stage_distribution_chart
screen_history_chart = chart_factory.screen_history_chart
sector_distribution_chart = chart_factory.sector_distribution_chart
backtest_horizon_chart = chart_factory.backtest_horizon_chart
backtest_path_chart = chart_factory.backtest_path_chart

print(inspect.signature(run_stealth_accumulation_screen))


(price_df: 'pd.DataFrame', volatility_df: 'pd.DataFrame | None' = None, distribution_df: 'pd.DataFrame | None' = None, market_df: 'pd.DataFrame | None' = None, config: 'dict[str, Any] | None' = None, benchmark_df: 'pd.DataFrame | None' = None) -> 'pd.DataFrame'


In [3]:
# Cell: Load Config And Market Data
config_path = "config/screens/stealth_accumulation.yaml"
screen_config = load_screen_config(config_path)

price_df = read_parquet("data/processed/clean_daily_ohlcv.parquet")
features_df = read_parquet("data/processed/stock_features.parquet")
volatility_df = features_df
distribution_df = features_df
market_df = read_parquet("data/processed/market_features.parquet")

coverage = {
    "symbols_in_price_data": int(price_df["symbol"].nunique()) if not price_df.empty else 0,
    "rows_in_price_data": int(len(price_df)),
    "date_min": str(pd.to_datetime(price_df["date"]).min().date()) if not price_df.empty else None,
    "date_max": str(pd.to_datetime(price_df["date"]).max().date()) if not price_df.empty else None,
    "market_feature_rows": int(len(market_df)),
    "has_market_entropy": bool("market_entropy_21d" in market_df.columns),
}
display(pd.DataFrame([coverage]).T.rename(columns={0: "value"}))
if not market_df.empty:
    display(market_df.sort_values("date").tail(3)[[col for col in ["date", "market_regime", "market_entropy_21d", "market_entropy_regime", "market_entropy_shift"] if col in market_df.columns]])


,value
symbols_in_price_data,500
rows_in_price_data,718709
date_min,2020-01-02
date_max,2026-07-22
market_feature_rows,1634
has_market_entropy,True


,date,market_regime,market_entropy_21d,market_entropy_regime,market_entropy_shift
1631,2026-07-20,Risk-off,0.700881,Transitional,Stable
1632,2026-07-21,Risk-off,0.734585,Transitional,Stable
1633,2026-07-22,Risk-off,0.734585,Transitional,Stable


In [4]:
# Cell: Set Optional Universe Filters
selected_symbols = []
selected_markets = []

# Vi du: selected_symbols = ["MBB", "SSI", "VCI"]
# Vi du: selected_markets = ["HOSE"]

if selected_symbols:
    screen_config["universe"]["symbols"] = selected_symbols
if selected_markets:
    screen_config["universe"]["markets"] = selected_markets

pd.DataFrame([
    {
        "selected_symbols_count": len(selected_symbols),
        "selected_markets_count": len(selected_markets),
    }
])


,selected_symbols_count,selected_markets_count
0,0,0


In [5]:
# Cell: Validate Scan Universe
scan_input = price_df.copy()
scan_mode = "full_local_market_dataset"
if selected_symbols:
    symbol_set = {str(symbol).upper() for symbol in selected_symbols}
    scan_input = scan_input[scan_input["symbol"].astype(str).str.upper().isin(symbol_set)].copy()
    scan_mode = "custom_symbol_subset"
if selected_markets and "market" in scan_input.columns:
    market_set = {str(market).upper() for market in selected_markets}
    scan_input = scan_input[scan_input["market"].astype(str).str.upper().isin(market_set)].copy()
    scan_mode = "custom_market_subset" if not selected_symbols else "custom_symbol_and_market_subset"

scan_summary = {
    "scan_mode": scan_mode,
    "raw_symbols_in_dataset": int(price_df["symbol"].nunique()) if not price_df.empty else 0,
    "effective_symbols_to_scan": int(scan_input["symbol"].nunique()) if not scan_input.empty else 0,
    "effective_rows_to_scan": int(len(scan_input)),
    "selected_symbol_count": len(selected_symbols),
    "selected_market_count": len(selected_markets),
}
display(pd.DataFrame([scan_summary]))
if not scan_input.empty and "market" in scan_input.columns:
    display(scan_input.groupby("market")["symbol"].nunique().rename("symbols").reset_index())


,scan_mode,raw_symbols_in_dataset,effective_symbols_to_scan,effective_rows_to_scan,selected_symbol_count,selected_market_count
0,full_local_market_dataset,500,500,718709,0,0


,market,symbols
0,HNX,69
1,HOSE,406
2,UPCOM,103


In [6]:
# Cell: Run Full Market Stealth Scan
screen_df = run_stealth_accumulation_screen(
    price_df=price_df,
    volatility_df=volatility_df,
    distribution_df=distribution_df,
    market_df=market_df,
    config=screen_config,
)

screen_df[[
    "symbol",
    "date",
    "accumulation_score",
    "breakout_quality_score",
    "fakeout_risk_score",
    "net_accumulation_score",
    "accumulation_stage",
    "screen_pass",
    "watchlist_flag",
]].tail(10)


,symbol,date,accumulation_score,breakout_quality_score,fakeout_risk_score,net_accumulation_score,accumulation_stage,screen_pass,watchlist_flag
637310,THD,2026-07-22,14.877503,22.931655,43.096459,27.397551,No Structure,False,False
637311,VTP,2026-07-22,14.069009,21.033503,40.810292,27.090307,No Structure,False,False
637312,VDP,2026-07-22,24.088908,11.560530,51.783260,26.988771,No Structure,False,False
637313,TRA,2026-07-22,19.327099,15.507974,47.851972,26.577550,No Structure,False,False
637314,NRC,2026-07-22,10.280013,20.641885,36.866260,26.083913,No Structure,False,False
637315,ACC,2026-07-22,10.526917,14.290316,32.704822,25.659832,No Structure,False,False
637316,TDC,2026-07-22,7.278547,17.465089,35.372381,24.162450,No Structure,False,False
637317,GIL,2026-07-22,2.531842,18.114438,40.342501,20.708905,No Structure,False,False
637318,BCE,2026-07-22,1.888953,17.332701,46.503817,18.651698,No Structure,False,False
637319,PNJ,2026-07-22,3.151672,23.857416,59.072023,17.772184,No Structure,False,False


In [7]:
# Cell: Review Raw Leaders
top_n = int(screen_config["output"]["top_n"])
sector_cfg = screen_config["sector"]

latest_leaders = latest_screen_snapshot(screen_df, top_n=top_n, only_watchlist=False)
sector_leaders = latest_screen_snapshot(
    screen_df,
    top_n=top_n,
    only_watchlist=False,
    sector_aware=True,
    max_names_per_sector=sector_cfg["max_names_per_sector"],
    sector_col=sector_cfg["column"],
    unknown_sector=sector_cfg["unknown_label"],
)

review_cols = [
    "symbol",
    "sector",
    "date",
    "accumulation_score",
    "breakout_quality_score",
    "fakeout_risk_score",
    "net_accumulation_score",
    "accumulation_stage",
    "screen_pass",
    "watchlist_flag",
]

display(latest_leaders[review_cols])
display(sector_leaders[review_cols + ["selection_rank"]] if "selection_rank" in sector_leaders.columns else sector_leaders[review_cols])


,symbol,sector,date,accumulation_score,breakout_quality_score,fakeout_risk_score,net_accumulation_score,accumulation_stage,screen_pass,watchlist_flag
0,MSB,Unclassified,2026-07-22,72.727722,53.876818,3.232323,74.024985,Stage S: Tight Squeeze,True,True
1,VPI,Unclassified,2026-07-22,65.320498,54.505483,15.745922,67.350139,Stage S: Tight Squeeze,True,True


,symbol,sector,date,accumulation_score,breakout_quality_score,fakeout_risk_score,net_accumulation_score,accumulation_stage,screen_pass,watchlist_flag,selection_rank
0,MSB,Unclassified,2026-07-22,72.727722,53.876818,3.232323,74.024985,Stage S: Tight Squeeze,True,True,1
1,VPI,Unclassified,2026-07-22,65.320498,54.505483,15.745922,67.350139,Stage S: Tight Squeeze,True,True,2


In [8]:
# Cell: Review Watchlist Candidates
latest_watchlist = latest_screen_snapshot(screen_df, top_n=top_n, only_watchlist=True)
sector_watchlist = latest_screen_snapshot(
    screen_df,
    top_n=top_n,
    only_watchlist=True,
    sector_aware=True,
    max_names_per_sector=sector_cfg["max_names_per_sector"],
    sector_col=sector_cfg["column"],
    unknown_sector=sector_cfg["unknown_label"],
)

display(latest_watchlist[review_cols])
display(sector_watchlist[review_cols + ["selection_rank"]] if not sector_watchlist.empty and "selection_rank" in sector_watchlist.columns else sector_watchlist.head(0))


,symbol,sector,date,accumulation_score,breakout_quality_score,fakeout_risk_score,net_accumulation_score,accumulation_stage,screen_pass,watchlist_flag
0,MSB,Unclassified,2026-07-22,72.727722,53.876818,3.232323,74.024985,Stage S: Tight Squeeze,True,True
1,VPI,Unclassified,2026-07-22,65.320498,54.505483,15.745922,67.350139,Stage S: Tight Squeeze,True,True


,symbol,sector,date,accumulation_score,breakout_quality_score,fakeout_risk_score,net_accumulation_score,accumulation_stage,screen_pass,watchlist_flag,selection_rank
0,MSB,Unclassified,2026-07-22,72.727722,53.876818,3.232323,74.024985,Stage S: Tight Squeeze,True,True,1
1,VPI,Unclassified,2026-07-22,65.320498,54.505483,15.745922,67.350139,Stage S: Tight Squeeze,True,True,2


In [9]:
# Cell: Visualize Scan Diagnostics
display(screen_score_bar_chart(latest_leaders, top_n=top_n, title="Full Market Stealth Leaders"))
display(screen_score_bar_chart(latest_watchlist if not latest_watchlist.empty else latest_leaders, top_n=top_n, title="Actionable Stealth Watchlist"))

latest_date = screen_df["date"].max()
display(stage_distribution_chart(screen_df[screen_df["date"] == latest_date], title="Latest Stage Distribution"))
display(sector_distribution_chart(sector_leaders, title="Sector Mix Of Sector-Aware Leaders"))


In [10]:
# Cell: Inspect Symbol History
focus_candidates = []
if not sector_watchlist.empty:
    focus_candidates.append(sector_watchlist["symbol"].iloc[0])
if not sector_leaders.empty:
    focus_candidates.append(sector_leaders["symbol"].iloc[0])
if not latest_leaders.empty:
    focus_candidates.append(latest_leaders["symbol"].iloc[0])
focus_symbol = focus_candidates[0] if focus_candidates else None

history_cols = [
    "date",
    "symbol",
    "sector",
    "accumulation_score",
    "breakout_quality_score",
    "fakeout_risk_score",
    "net_accumulation_score",
    "accumulation_stage",
    "distance_to_pivot",
    "volume_ratio_5_20",
    "relative_strength_20d",
]

if focus_symbol is None:
    print("Chua co symbol nao vuot qua bo loc hien tai de inspect history")
else:
    display(screen_df[screen_df["symbol"] == focus_symbol][history_cols].tail(20))
    screen_history_chart(screen_df, symbol=focus_symbol, title="Stealth Accumulation Score History")


,date,symbol,sector,accumulation_score,breakout_quality_score,fakeout_risk_score,net_accumulation_score,accumulation_stage,distance_to_pivot,volume_ratio_5_20,relative_strength_20d
629551,2026-06-25,MSB,Unclassified,56.359974,75.381307,8.027211,70.018511,No Structure,-0.009434,0.710982,0.043368
629965,2026-06-26,MSB,Unclassified,53.555708,75.627465,7.687075,68.762952,No Structure,0.000000,0.620618,0.034697
630342,2026-06-29,MSB,Unclassified,57.711769,79.011660,7.272727,71.790618,Stage A: Silent Accumulation,0.006289,0.712767,0.117152
630723,2026-06-30,MSB,Unclassified,58.024142,80.055595,0.000000,74.025970,Stage A: Silent Accumulation,0.003096,0.735512,0.114504
631161,2026-07-01,MSB,Unclassified,55.003820,83.964090,16.363636,69.402023,Stage A: Silent Accumulation,-0.012346,0.721940,0.076950
631560,2026-07-02,MSB,Unclassified,53.720285,77.019241,18.181818,66.569498,No Structure,-0.015432,0.803908,0.077220
631907,2026-07-03,MSB,Unclassified,53.301212,78.736896,0.000000,71.334830,No Structure,-0.012346,0.804730,0.068476
632307,2026-07-06,MSB,Unclassified,57.949731,69.212213,11.818182,68.323373,Stage A: Silent Accumulation,-0.024691,0.864731,0.083093
632668,2026-07-07,MSB,Unclassified,60.591236,80.487077,0.000000,75.417387,Stage C: Valid Expansion,-0.003086,0.770221,0.083008
633096,2026-07-08,MSB,Unclassified,58.646287,83.458410,20.000000,70.187746,Stage A: Silent Accumulation,-0.012346,0.759993,0.068175


In [11]:
# Cell: Backtest Screener Quality
backtest_cfg = screen_config["backtest"]
backtest_results = compare_screen_backtests(
    screen_df,
    top_n=backtest_cfg["top_n"],
    holding_periods=backtest_cfg["holding_periods"],
    only_watchlist=backtest_cfg["only_watchlist"],
    max_names_per_sector=sector_cfg["max_names_per_sector"],
    sector_col=sector_cfg["column"],
    unknown_sector=sector_cfg["unknown_label"],
    benchmark_symbol=backtest_cfg.get("benchmark_symbol", "VNINDEX"),
    benchmark_path=backtest_cfg.get("benchmark_path"),
)

display(backtest_results["summary"])
display(backtest_horizon_chart(backtest_results["summary"], title="Raw Vs Sector-Aware Forward Returns"))
display(backtest_horizon_chart(backtest_results["summary"], value_col="avg_excess_return", title="Raw Vs Sector-Aware Excess Returns"))
display(backtest_path_chart(backtest_results["by_date"], holding_period=10, title="10D Forward Return History"))


,portfolio_type,holding_period_days,observations,avg_forward_return,avg_net_forward_return,avg_trading_cost,avg_signal_to_entry_gap,avg_benchmark_return,avg_excess_return,avg_net_excess_return,...,excess_hit_rate,net_excess_hit_rate,avg_selection_count,volatility_of_avg_returns,volatility_of_avg_net_returns,sharpe_like,net_sharpe_like,information_ratio_like,net_information_ratio_like,benchmark_source
0,raw,5,1020,0.001319,0.001319,0.0,0.000407,0.000922,0.000397,0.000397,...,0.487255,0.487255,4.351961,0.034560,0.034560,0.270964,0.270964,0.099057,0.099057,benchmark_vnindex.parquet
1,raw,10,1015,0.003411,0.003411,0.0,0.000429,0.002965,0.000446,0.000446,...,0.492611,0.492611,4.348768,0.048386,0.048386,0.353926,0.353926,0.053831,0.053831,benchmark_vnindex.parquet
2,raw,20,1005,0.006658,0.006658,0.0,0.000440,0.005826,0.000832,0.000832,...,0.480597,0.480597,4.342289,0.069181,0.069181,0.341602,0.341602,0.053052,0.053052,benchmark_vnindex.parquet
3,sector_aware,5,1020,0.001619,0.001619,0.0,0.000277,0.000922,0.000697,0.000697,...,0.502941,0.502941,3.268627,0.032676,0.032676,0.351820,0.351820,0.172418,0.172418,benchmark_vnindex.parquet
4,sector_aware,10,1015,0.003351,0.003351,0.0,0.000301,0.002965,0.000386,0.000386,...,0.509360,0.509360,3.269951,0.045981,0.045981,0.365890,0.365890,0.046298,0.046298,benchmark_vnindex.parquet
5,sector_aware,20,1005,0.003391,0.003391,0.0,0.000316,0.005826,-0.002435,-0.002435,...,0.458706,0.458706,3.280597,0.066626,0.066626,0.180664,0.180664,-0.152633,-0.152633,benchmark_vnindex.parquet


In [12]:
# Cell: Review Market Entropy Context
if market_df.empty or "market_entropy_21d" not in market_df.columns:
    print("Market entropy chua san sang. Chay notebook 06 de build them entropy profile cho market.")
else:
    display(entropy_regime_chart(market_df.sort_values("date"), entropy_col="market_entropy_21d", regime_col="market_entropy_regime", title="VN Market Entropy Context"))
    display(market_df.sort_values("date").tail(10)[[col for col in ["date", "market_regime", "market_entropy_21d", "market_entropy_63d", "market_entropy_regime", "market_entropy_shift", "advance_ratio"] if col in market_df.columns]])


,date,market_regime,market_entropy_21d,market_entropy_63d,market_entropy_regime,market_entropy_shift,advance_ratio
1624,2026-07-09,Risk-off,0.742425,0.830062,Transitional,Stable,0.248837
1625,2026-07-10,Risk-off,0.747516,0.827103,Transitional,Stable,0.236239
1626,2026-07-13,Risk-off,0.713813,0.857958,Transitional,Stable,0.140909
1627,2026-07-14,Neutral,0.776128,0.856617,Transitional,Stable,0.513953
1628,2026-07-15,Risk-off,0.771037,0.872297,Transitional,Stable,0.195755
1629,2026-07-16,Risk-off,0.783968,0.872297,Transitional,Stable,0.392111
1630,2026-07-17,Risk-off,0.776128,0.868332,Transitional,Stable,0.259091
1631,2026-07-20,Risk-off,0.700881,0.755409,Transitional,Stable,0.114679
1632,2026-07-21,Risk-off,0.734585,0.771646,Transitional,Stable,0.271071
1633,2026-07-22,Risk-off,0.734585,0.779776,Transitional,Stable,0.200000


In [13]:
# Cell: Save Scan Outputs
save_parquet(screen_df, "data/signals/stealth_accumulation_history.parquet")
save_parquet(latest_leaders, "data/signals/stealth_accumulation_latest.parquet")
save_parquet(latest_watchlist, "data/signals/stealth_accumulation_watchlist.parquet")
save_parquet(sector_leaders, "data/signals/stealth_accumulation_sector_latest.parquet")
save_parquet(sector_watchlist, "data/signals/stealth_accumulation_sector_watchlist.parquet")
save_parquet(backtest_results["summary"], "data/signals/stealth_accumulation_backtest_summary.parquet")
save_parquet(backtest_results["by_date"], "data/signals/stealth_accumulation_backtest_by_date.parquet")
save_parquet(backtest_results["selections"], "data/signals/stealth_accumulation_backtest_selections.parquet")

print("Saved stealth accumulation outputs to data/signals")


Saved stealth accumulation outputs to data/signals
